In [233]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import os
import plotly.express as px

In [234]:
results = pd.read_csv("data/processed/world_cup/all_editions/results.csv")
history = pd.read_csv("INT-World Cup/world_cup/fifa_world_cup_history.csv")
placement = pd.read_csv("data/processed/world_cup/all_editions/placement.csv")
all_teams = pd.read_csv("data/processed/world_cup/all_editions/teams.csv")
teams = pd.read_csv("data/processed/world_cup/2026/teams.csv")
pd.set_option("display.max_columns", None)

In [235]:
#Add next edition placement for all countries
#print(history.columns)
#print(placement.columns)
print(teams.columns)

history = history.rename(columns={"Year": "edition", "Teams": "teams"})
placement = placement.merge(
   history[["edition", "teams"]],
    on="edition",
    how="left",
    validate="many_to_one"
    )

#Add eras
era_bins = [1929, 1950, 1970, 1990, 2010, 2026]
era_labels = [
    "Early Era",
    "Golden Age",
    "Modern Era",
    "Contemporary",
    "Recent",
]

teams["era"] = pd.cut(
    teams["year"],
    bins=era_bins,
    labels=era_labels,
)

all_teams["era"] = pd.cut(
    all_teams["year"],
    bins=era_bins,
    labels=era_labels,
)

print(teams.columns)
print(all_teams.columns)

#display(history)

Index(['tournament_id', 'year', 'team_id', 'team', 'canonical_name',
       'tournament_name', 'fifa_code', 'flag_icon_code', 'flag_icon_css_class',
       'confederation', 'group_code', 'is_host', 'qualification_path',
       'world_cup_participations', 'weighted_world_cup_participations',
       'weighted_world_cup_placement_score', 'squad_status', 'source_url',
       'source_as_of'],
      dtype='object')
Index(['tournament_id', 'year', 'team_id', 'team', 'canonical_name',
       'tournament_name', 'fifa_code', 'flag_icon_code', 'flag_icon_css_class',
       'confederation', 'group_code', 'is_host', 'qualification_path',
       'world_cup_participations', 'weighted_world_cup_participations',
       'weighted_world_cup_placement_score', 'squad_status', 'source_url',
       'source_as_of', 'era'],
      dtype='object')
Index(['tournament_id', 'year', 'team_id', 'team', 'team_code',
       'confederation', 'tournament_name', 'placement', 'position',
       'matches_played', 'era'],
  

In [236]:
#display(teams.describe)
print(all_teams.isna().sum())

#Assign 

tournament_id       0
year                0
team_id             0
team                0
team_code           0
confederation       0
tournament_name     0
placement           0
position           48
matches_played     48
era                 0
dtype: int64


## Distribution of 2026 FIFA Men's World Cup Teams by Confederation


Ahead of the 2026 FIFA Men's World Cup, the tournament size was expanded once again from 32 teams -> 48 teams, and with this expansion saw confederations receive major slots increase like CAF (increase from 5 -> 10 spots)

In [237]:
#tree map of all the represented nations by confederation

teams_df = teams.copy()
CONFEDERATION_COLORS = {
    "UEFA":     "#1A56DB",  # Strong Blue
    "CAF":      "#F5A623",  # Warm Gold
    "AFC":      "#E02020",  # Vivid Red
    "CONMEBOL": "#27AE60",  # Emerald Green
    "CONCACAF": "#8E44AD",  # Purple (distinct from others)
    "OFC":      "#17A8CD",  # Sky/Ocean Blue
}


fig = px.treemap(
    teams_df,
    path=[px.Constant("2026 FIFA Men's World Cup"), 'confederation', 'team'],
    color="confederation",
    color_discrete_map=CONFEDERATION_COLORS
    )
fig.update_layout(
    margin = dict(t=50, l=25, r=25, b=25), 
    height=800, 
    width=800,
    title=dict(
        text=f"Distribution of 2026 FIFA Men's World Cup Teams by Confederation",
        x=0.5,
        font=dict(size=20, color="#3A2A1A"),
    ),
    font=dict(
        family="Inter, sans-serif",
        color="#3A2A1A",
        size=11,
    ),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
)
fig.update_traces(marker=dict(cornerradius=5))

fig.add_annotation(
        text="Data Source: Kaggle | @cartierkut1",
        x=0, y=-0.03,
        showarrow=False,
        font=dict(size=10, color="#5A4632"),
)
fig.show(scale=3)


In [238]:
#Participations per confederation at every wc
participation = all_teams.copy()
print(participation.columns)

p_sum = (
    participation
    .dropna(subset=["confederation", "era"])
    .groupby(["year", "era", "confederation"], as_index=False, observed=True)
    .agg(cs=("team", "nunique"))
    .sort_values(["year", "confederation"])
    .reset_index(drop=True)
)



print(p_sum.columns)

selected_confederation = "CAF"

country_df = (
    p_sum.query("confederation == @selected_confederation")
    .sort_values("year")
    .copy()
)

maps_df = (
    teams_df.query("confederation == @selected_confederation")
    .sort_values("year")
    .copy()
)

tick_vals = sorted(country_df["year"].unique())

confederation_name = selected_confederation

era_colors = {
    "Early Era": "#7A4E2D",
    "Golden Age": "#C99700",
    "Modern Era": "#2F6F73",
    "Contemporary": "#8A3FFC",
    "Recent": "#D1495B",
}


fig = px.bar(
    country_df,
    x="year",
    y="cs",
    color="era",
    color_discrete_map=era_colors,
    text='cs',    
)
fig.update_layout(
    height=600,
    width=1000,
    title=dict(
        text=f"{confederation_name}'s Team count at every FIFA Men's World Cup",
        x=0.5,
        font=dict(size=20, color="#3A2A1A"),
    ),
    coloraxis_colorbar=dict(title="Goals Rank"),
    font=dict(
        family="Gill Sans, Arial, sans-serif",
        color="#3A2A1A",
        size=11,
    ),
    margin=dict(l=40, r=40, t=70, b=60),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    xaxis=dict(
        title="Tournament Edition",
        tickmode="array",
        tickvals=tick_vals,
        ticktext=[str(x) for x in tick_vals],
        showgrid=False,
        zeroline=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
        tickangle=30,
    ),
    yaxis=dict(
        title="Number of teams",
        showgrid=False,
        zeroline=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
    ),
)

if selected_confederation == "CAF" and 2010 in country_df["year"].values:
    south_africa_team_count = country_df.loc[country_df["year"].eq(2010), "cs"].iloc[0]
    fig.add_annotation(
        x=2010,
        y=south_africa_team_count,
        text="South Africa World Cup",
        showarrow=True,
        arrowhead=1,
        ax=45,
        ay=-30,
        font=dict(size=12, color="#3A2A1A"),
        arrowcolor="#5A4632",
    )

if selected_confederation == "CAF" and 1934 in country_df["year"].values:
    first_african_team_count = country_df.loc[country_df["year"].eq(1934), "cs"].iloc[0]
    first_african_team_rows = participation.loc[
        participation["confederation"].eq("CAF") & participation["year"].eq(1934),
        "team",
    ]
    first_african_team = first_african_team_rows.iloc[0] if not first_african_team_rows.empty else "Egypt"
    fig.add_annotation(
        x=1934,
        y=first_african_team_count,
        text=f"{first_african_team}: first African team",
        showarrow=True,
        arrowhead=1,
        ax=45,
        ay=-30,
        font=dict(size=12, color="#3A2A1A"),
        arrowcolor="#5A4632",
    )


fig.add_annotation(
    text="Data Source: Kaggle | @cartierkut1",
    xref="paper",
    yref="paper",
    x=0,
    y=-0.13,
    showarrow=False,
    font=dict(size=10, color="#5A4632"),
)



fig.show(scale=3)


Index(['tournament_id', 'year', 'team_id', 'team', 'team_code',
       'confederation', 'tournament_name', 'placement', 'position',
       'matches_played', 'era'],
      dtype='object')
Index(['year', 'era', 'confederation', 'cs'], dtype='object')


In [239]:
scope_map = {
    "UEFA": "europe",
    "CAF": "africa",
    "AFC": "asia",
    "CONMEBOL": "south america",
    "CONCACAF": "north america",
    "OFC": "world",  # Plotly has no clean Oceania scope
}

plot_df = maps_df[maps_df["confederation"].eq(selected_confederation)].copy()

fig3 = px.choropleth(
    plot_df,
    locations="fifa_code",      # or your ISO-3 column
    locationmode="ISO-3",
    color="team",
    hover_name="team",
    scope=scope_map[selected_confederation],
    title=f"{selected_confederation} World Cup Teams"
)

fig3.show()

In [240]:
#Participations per confederation at every wc
participation = all_teams.copy()
print(participation.columns)

p_sum = (
    participation
    .dropna(subset=["confederation", "era"])
    .groupby(["year", "era", "confederation"], as_index=False, observed=True)
    .agg(cs=("team", "nunique"))
    .sort_values(["year", "confederation"])
    .reset_index(drop=True)
)



print(p_sum.columns)

selected_confederation = "UEFA"

country_df = (
    p_sum.query("confederation == @selected_confederation")
    .sort_values("year")
    .copy()
)

tick_vals = sorted(country_df["year"].unique())

confederation_name = selected_confederation

era_colors = {
    "Early Era": "#7A4E2D",
    "Golden Age": "#C99700",
    "Modern Era": "#2F6F73",
    "Contemporary": "#8A3FFC",
    "Recent": "#D1495B",
}


fig = px.bar(
    p_sum,
    x="year",
    y="cs",
    color="confederation",
    text='cs',   
)

fig.update_layout(
    height=600,
    width=1000,
    title=dict(
        text=f"{confederation_name}'s Participation in Every World Cup",
        x=0.5,
        font=dict(size=20, color="#3A2A1A"),
    ),
    coloraxis_colorbar=dict(title="Goals Rank"),
    font=dict(
        family="Gill Sans, Arial, sans-serif",
        color="#3A2A1A",
        size=11,
    ),
    margin=dict(l=40, r=40, t=70, b=60),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    xaxis=dict(
        title="Tournament Edition",
        tickmode="array",
        tickvals=tick_vals,
        ticktext=[str(x) for x in tick_vals],
        showgrid=False,
        zeroline=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
        tickangle=30,
    ),
    yaxis=dict(
        title="Goals Scored",
        showgrid=False,
        zeroline=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
    ),
)

fig.update_traces(
    textposition='inside'
)
fig.add_annotation(
    text="Data Source: Kaggle | @cartierkut1",
    xref="paper",
    yref="paper",
    x=0,
    y=-0.13,
    showarrow=False,
    font=dict(size=10, color="#5A4632"),
)

fig.show(scale=3)


Index(['tournament_id', 'year', 'team_id', 'team', 'team_code',
       'confederation', 'tournament_name', 'placement', 'position',
       'matches_played', 'era'],
      dtype='object')
Index(['year', 'era', 'confederation', 'cs'], dtype='object')


In [241]:
#Add Team and Opponent columns to the results csv
results_df = results.copy()



results_df = results_df.assign(
    GF=results_df["team_score"],
    GA=results_df["opponent_score"],
)



eras = {
    "Early Era": [1930, 1934, 1938],
    "Golden Age": [1950, 1954, 1958, 1962, 1966, 1970],
    "Modern Era": [1974, 1978, 1982, 1986, 1990],
    "Contemporary": [1994, 1998, 2002, 2006, 2010],
    "Recent": [2014, 2018, 2022],
}

edition_to_era = {
    edition: era
    for era, editions in eras.items()
    for edition in editions
}

results_df["era"] = results_df["edition"].map(edition_to_era)

display(results_df.head())

,edition,tournament_id,match_id,match_number,date,stage,status,team_id,team,team_confederation,opponent_id,opponent,opponent_confederation,is_home,team_score,opponent_score,result,team_elo_start,opponent_elo_start,team_elo_end,opponent_elo_end,team_elo_delta,city,country,neutral,decided_by_shootout,shootout_winner,GF,GA,era
0,1930,WC-1930,WC-1930_001,1,1930-07-13,Group Stage,played,BEL,Belgium,UEFA,USA,United States,CONCACAF,True,0,3,loss,1661.0,1672.0,1610.0,1723.0,-51.0,Montevideo,Uruguay,True,False,NaN,0,3,Early Era
1,1930,WC-1930,WC-1930_001,1,1930-07-13,Group Stage,played,USA,United States,CONCACAF,BEL,Belgium,UEFA,False,3,0,win,1672.0,1661.0,1723.0,1610.0,51.0,Montevideo,Uruguay,True,False,NaN,3,0,Early Era
2,1930,WC-1930,WC-1930_002,2,1930-07-13,Group Stage,played,FRA,France,UEFA,MEX,Mexico,CONCACAF,True,4,1,win,1547.0,1613.0,1609.0,1551.0,62.0,Montevideo,Uruguay,True,False,NaN,4,1,Early Era
3,1930,WC-1930,WC-1930_002,2,1930-07-13,Group Stage,played,MEX,Mexico,CONCACAF,FRA,France,UEFA,False,1,4,loss,1613.0,1547.0,1551.0,1609.0,-62.0,Montevideo,Uruguay,True,False,NaN,1,4,Early Era
4,1930,WC-1930,WC-1930_003,3,1930-07-14,Group Stage,played,BRA,Brazil,CONMEBOL,YUG,Yugoslavia,UEFA,True,1,2,loss,1923.0,1590.0,1871.0,1642.0,-52.0,Montevideo,Uruguay,True,False,NaN,1,2,Early Era


In [242]:
#Add goal tally across World Cups
placement_score_map = {
    "Winner": 7,
    "Runner-up": 6,
    "Third Place": 5,
    "Fourth Place": 4,
    "Quarter-final": 3,
    "Round of 16": 2,
    "Group Stage": 1,
}

placement["placement_score"] = placement["placement"].map(placement_score_map)

display(placement[["edition", "country", "placement", "position", "placement_score"]])

gls = (
    results_df.groupby(["edition", "era", "tournament_id", "team"], dropna=False, as_index=False)
    .agg(
        gf=("GF", "sum"),
        ga=("GA", "sum"),
    )
)

gls["gls_rank"] = (
    gls.groupby(["edition", "tournament_id"])["gf"]
    .rank(method="min", ascending=False)
    .astype(int)
)

gls["ga_rank"] = (
    gls.groupby(["edition", "tournament_id"])["ga"]
    .rank(method="min", ascending=True)
    .astype(int)
)

gls = gls.sort_values(["edition", "team"]).reset_index(drop=True)
gls = gls.rename(columns={"team": "country"})

#Add placements
gls = gls.merge(
    placement[["country", "placement", "edition", "position"]],
                on=["country", "edition"],
                how="left")

display(gls)



,edition,country,placement,position,placement_score
0,1930,Uruguay,Winner,1,7.0
1,1930,Argentina,Runner-up,2,6.0
2,1930,United States,Third Place,3,5.0
3,1930,Yugoslavia,Fourth Place,4,4.0
4,1930,Belgium,Group Stage,13,1.0
...,...,...,...,...,...
484,2022,Saudi Arabia,Group Stage,32,1.0
485,2022,Serbia,Group Stage,32,1.0
486,2022,Tunisia,Group Stage,32,1.0
487,2022,Uruguay,Group Stage,32,1.0


,edition,era,tournament_id,country,gf,ga,gls_rank,ga_rank,placement,position
0,1930,Early Era,WC-1930,Argentina,18,9,1,12,Runner-up,2
1,1930,Early Era,WC-1930,Belgium,0,4,12,6,Group Stage,13
2,1930,Early Era,WC-1930,Bolivia,0,8,12,11,Group Stage,13
3,1930,Early Era,WC-1930,Brazil,5,2,5,1,Group Stage,13
4,1930,Early Era,WC-1930,Chile,5,3,5,2,Group Stage,13
...,...,...,...,...,...,...,...,...,...,...
484,2022,Recent,WC-2022,Switzerland,5,9,11,31,Round of 16,16
485,2022,Recent,WC-2022,Tunisia,1,1,28,1,Group Stage,32
486,2022,Recent,WC-2022,United States,3,4,21,9,Round of 16,16
487,2022,Recent,WC-2022,Uruguay,2,2,25,2,Group Stage,32


In [243]:
selected_country = "Argentina"

country_df = (
    gls.query("country == @selected_country")
    .sort_values("edition")
    .copy()
)

tick_vals = sorted(country_df["edition"].unique())

country_name = selected_country

era_colors = {
    "Early Era": "#7A4E2D",
    "Golden Age": "#C99700",
    "Modern Era": "#2F6F73",
    "Contemporary": "#8A3FFC",
    "Recent": "#D1495B",
}


fig = px.bar(
    country_df,
    x="edition",
    y="gf",
    color="era",    
    color_discrete_map=era_colors,
    hover_data={
        "country": True,
        "placement": True,
        "gf": True,
        "gls_rank": True,
        "edition": False,
    },
    labels={
        "edition": "Tournament Edition",
        "gf": "Goals Scored",
        "gls_rank": "Goals Rank",
        "placement": "Final Placement",
    },
    color_continuous_scale="Viridis_r",
    text="gf"
)

fig.update_layout(
    height=600,
    width=1000,
    title=dict(
        text=f"{country_name}'s Goals Scored in Every World Cup Participation",
        x=0.5,
        font=dict(size=20, color="#3A2A1A"),
    ),
    coloraxis_colorbar=dict(title="Goals Rank"),
    font=dict(
        family="Gill Sans, Arial, sans-serif",
        color="#3A2A1A",
        size=11,
    ),
    margin=dict(l=40, r=40, t=70, b=60),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    xaxis=dict(
        title="Tournament Edition",
        tickmode="array",
        tickvals=tick_vals,
        ticktext=[str(x) for x in tick_vals],
        showgrid=False,
        zeroline=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
    ),
    yaxis=dict(
        title="Goals Scored",
        showgrid=False,
        zeroline=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
    ),
)

fig.update_traces(
    textposition='outside'
)
fig.add_annotation(
    text="Data Source: Kaggle | @cartierkut1",
    xref="paper",
    yref="paper",
    x=0,
    y=-0.13,
    showarrow=False,
    font=dict(size=10, color="#5A4632"),
)

fig.show(scale=3)


In [244]:
goals_by_tournament = (
    gls.groupby("edition", as_index=False)
    .agg(total_goals=("gf", "sum"))
)
tick_vals = sorted(goals_by_tournament["edition"].unique())


goals_by_tournament["edition"] = goals_by_tournament["edition"].astype(int)

era_bins = [1929, 1950, 1970, 1990, 2010, 2026]
era_labels = [
    "Early Era",
    "Golden Age",
    "Modern Era",
    "Contemporary",
    "Recent",
]

goals_by_tournament["era"] = pd.cut(
    goals_by_tournament["edition"],
    bins=era_bins,
    labels=era_labels,
)

era_colors = {
    "Early Era": "#7A4E2D",
    "Golden Age": "#C99700",
    "Modern Era": "#2F6F73",
    "Contemporary": "#8A3FFC",
    "Recent": "#D1495B",
}

goals_by_tournament["era"] = goals_by_tournament["edition"].map(edition_to_era)

display(goals_by_tournament)
fig = px.bar(
    goals_by_tournament,
    x="edition",
    y="total_goals",
    color="era",
    color_discrete_map=era_colors,
    text='total_goals'
)


fig.update_layout(
    height=600,
    width=1000,
     title=dict(
            font=dict(size=20, color="#3A2A1A"),
            text= f"Total goals scored at every World Cup",
            x=0.5
        ),
    legend = dict(
        title="Team"
        ),
    font=dict(
        family="Gill Sans, Arial, sans-serif",
        color="#3A2A1A",
        size=10,
    ),
    margin=dict(l=40, r=40, t=60, b=40),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    xaxis=dict(
        title="Tournament Edition",
        showgrid=False,
        gridcolor='rgba(0,255,65,0.15)',
        zeroline=False,
        linecolor='#5A4632',
        tickfont=dict(size=13, color="#5A4632"),
        tickmode="array",
        tickvals=tick_vals,
        ticktext=[str(x) for x in tick_vals],
    ),

    yaxis=dict(
        title="Goals Scored",
        showgrid=False,
        gridcolor='rgba(0,255,65,0.15)',
        zeroline=False,
        linecolor='#5A4632',
        tickfont=dict(size=13, color="#5A4632"),
    )
        
)

fig.update_traces(
    textposition='inside'
)
#add caption at the bottom of the chart
fig.add_annotation(
        text="Data Source: Kaggle | @cartierkut1",
        xref="paper", yref="paper",
        x=0, y=-0.1,
        showarrow=False,
        font=dict(size=10, color="#5A4632"),
)


fig.show(scale=3)

,edition,total_goals,era
0,1930,70,Early Era
1,1934,70,Early Era
2,1938,84,Early Era
3,1950,88,Golden Age
4,1954,140,Golden Age
5,1958,126,Golden Age
6,1962,89,Golden Age
7,1966,89,Golden Age
8,1970,95,Golden Age
9,1974,97,Modern Era


In [245]:
# World Cup outcome correlation analysis
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

placement_score_map = {
    "Winner": 7,
    "Runner-up": 6,
    "Third Place": 5,
    "Fourth Place": 4,
    "Semi-final": 4,
    "Quarter-final": 3,
    "Round of 16": 2,
    "Group Stage": 1,
}

placement_files = sorted(Path("data/processed/world_cup").glob("[0-9][0-9][0-9][0-9]/placement.csv"))
placement_frames = []
for placement_file in placement_files:
    edition = int(placement_file.parent.name)
    edition_df = pd.read_csv(placement_file)
    edition_df["edition"] = edition
    placement_frames.append(edition_df)

outcome_df = pd.concat(placement_frames, ignore_index=True)
outcome_df = outcome_df.sort_values(["country", "edition"]).reset_index(drop=True)

# Outcome: higher is better, normalized within each tournament size.
outcome_df["edition_team_count"] = outcome_df.groupby("edition")["country"].transform("count")
outcome_df["position"] = pd.to_numeric(outcome_df["position"], errors="coerce")
outcome_df["edition_finish_scale"] = outcome_df.groupby("edition")["position"].transform("max")
outcome_df["edition_finish_scale"] = outcome_df[["edition_team_count", "edition_finish_scale"]].max(axis=1)
outcome_df["finish_score"] = 1 - (
    (outcome_df["position"] - 1) / (outcome_df["edition_finish_scale"] - 1)
)
outcome_df["placement_score"] = outcome_df["placement"].map(placement_score_map)

# In-tournament explanatory features. These describe outcomes but should not be treated as pre-tournament predictors.
for column in ["matches_played", "gs", "ga", "start_elo", "finish_elo", "elo_change"]:
    outcome_df[column] = pd.to_numeric(outcome_df[column], errors="coerce")

outcome_df["goal_difference"] = outcome_df["gs"] - outcome_df["ga"]
outcome_df["goals_per_match"] = outcome_df["gs"] / outcome_df["matches_played"]
outcome_df["goals_against_per_match"] = outcome_df["ga"] / outcome_df["matches_played"]
outcome_df["goal_difference_per_match"] = outcome_df["goal_difference"] / outcome_df["matches_played"]

# Host flag from the match results host-country column. Use pairs so co-host editions do not duplicate rows.
host_pairs = set(
    pd.read_csv("data/processed/world_cup/all_editions/results.csv")[["edition", "country"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
outcome_df["is_host"] = [
    int((int(edition), country) in host_pairs)
    for edition, country in zip(outcome_df["edition"], outcome_df["country"])
]

# Leakage-safe prior features: every rolling value is shifted so the current tournament is excluded.
team_history = outcome_df.groupby("country", group_keys=False)
outcome_df["prior_world_cup_participations"] = team_history.cumcount()
outcome_df["previous_finish_score"] = team_history["finish_score"].shift(1)
outcome_df["previous_position"] = team_history["position"].shift(1)

outcome_df["prior_avg_finish_score"] = team_history["finish_score"].transform(
    lambda series: series.shift(1).expanding().mean()
)
outcome_df["prior_best_finish_score"] = team_history["finish_score"].transform(
    lambda series: series.shift(1).expanding().max()
)
outcome_df["prior_avg_goals_per_match"] = team_history["goals_per_match"].transform(
    lambda series: series.shift(1).expanding().mean()
)
outcome_df["prior_avg_goal_diff_per_match"] = team_history["goal_difference_per_match"].transform(
    lambda series: series.shift(1).expanding().mean()
)

# Leakage-safe last-k form entering each tournament.
FORM_MATCH_WINDOW = 10
country_form_aliases = {
    "China PR": "China",
    "DR Congo": "Zaire",
}

edition_start_dates = {}
for results_file in sorted(Path("data/processed/world_cup").glob("[0-9][0-9][0-9][0-9]/results.csv")):
    edition = int(results_file.parent.name)
    edition_results = pd.read_csv(results_file, usecols=["date"])
    edition_start_dates[edition] = pd.to_datetime(edition_results["date"], errors="coerce").min()

team_result_frames = []
team_result_columns = [
    "date",
    "team",
    "team_score",
    "opponent_score",
    "result",
    "team_elo_start",
    "opponent_elo_start",
    "team_elo_delta",
]
for team_results_file in sorted(Path("data/processed/world_cup/by_confederation").glob("*/*/results.csv")):
    team_results = pd.read_csv(team_results_file, usecols=team_result_columns)
    if team_results.empty:
        continue
    team_result_frames.append(team_results)

team_results_history = pd.concat(team_result_frames, ignore_index=True)
team_results_history["date"] = pd.to_datetime(team_results_history["date"], errors="coerce")
for column in ["team_score", "opponent_score", "team_elo_start", "opponent_elo_start", "team_elo_delta"]:
    team_results_history[column] = pd.to_numeric(team_results_history[column], errors="coerce")

normalized_result = (
    team_results_history["result"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({"w": "win", "d": "draw", "l": "loss", "win": "win", "draw": "draw", "loss": "loss"})
)
score_based_result = pd.Series(
    np.select(
        [
            team_results_history["team_score"] > team_results_history["opponent_score"],
            team_results_history["team_score"] == team_results_history["opponent_score"],
            team_results_history["team_score"] < team_results_history["opponent_score"],
        ],
        ["win", "draw", "loss"],
        default=None,
    ),
    index=team_results_history.index,
    dtype="object",
)
team_results_history["normalized_result"] = normalized_result.fillna(score_based_result)
team_results_history["actual_score"] = team_results_history["normalized_result"].map({"win": 1.0, "draw": 0.5, "loss": 0.0})
team_results_history["win_indicator"] = team_results_history["normalized_result"].eq("win").astype(float)
team_results_history["goal_difference"] = team_results_history["team_score"] - team_results_history["opponent_score"]
team_results_history = team_results_history.dropna(subset=["date", "team", "team_score", "opponent_score", "actual_score"]).copy()
team_results_lookup = {
    str(team): matches.sort_values("date", kind="stable").reset_index(drop=True)
    for team, matches in team_results_history.groupby("team", sort=False)
}

def weighted_mean(values, weights):
    values = pd.to_numeric(values, errors="coerce")
    valid = values.notna()
    if not valid.any():
        return np.nan
    return float(np.average(values[valid], weights=weights[valid]))

form_rows = []
for row in outcome_df[["edition", "country"]].itertuples(index=False):
    edition_start_date = edition_start_dates.get(int(row.edition))
    lookup_country = country_form_aliases.get(str(row.country), str(row.country))
    team_matches = team_results_lookup.get(lookup_country)
    if team_matches is None or pd.isna(edition_start_date):
        recent = pd.DataFrame(columns=team_results_history.columns)
    else:
        recent = team_matches[team_matches["date"] < edition_start_date].tail(FORM_MATCH_WINDOW).copy()

    form_row = {
        "edition": int(row.edition),
        "country": str(row.country),
        "form_l10_matches": np.nan,
        "form_l10_win_pct": np.nan,
        "form_l10_goals_for": np.nan,
        "form_l10_goals_against": np.nan,
        "form_l10_goal_difference": np.nan,
        "form_l10_goals_per_match": np.nan,
        "form_l10_goals_against_per_match": np.nan,
        "form_l10_goal_difference_per_match": np.nan,
        "form_l10_elo_change": np.nan,
        "form_l10_elo_change_per_match": np.nan,
        "weighted_form_l10_result_score": np.nan,
        "weighted_form_l10_win_pct": np.nan,
        "weighted_form_l10_goals_for_per_match": np.nan,
        "weighted_form_l10_goals_against_per_match": np.nan,
        "weighted_form_l10_goal_difference_per_match": np.nan,
        "weighted_form_l10_elo_change_per_match": np.nan,
        "form_l10_first_match_date": pd.NaT,
        "form_l10_last_match_date": pd.NaT,
    }

    if not recent.empty:
        match_count = len(recent)
        weights = pd.Series(np.arange(1, match_count + 1, dtype=float), index=recent.index)
        goals_for = recent["team_score"].sum()
        goals_against = recent["opponent_score"].sum()
        goal_difference = goals_for - goals_against
        elo_change = recent["team_elo_delta"].sum(min_count=1)

        form_row.update(
            {
                "form_l10_matches": int(match_count),
                "form_l10_win_pct": float(recent["win_indicator"].mean()),
                "form_l10_goals_for": float(goals_for),
                "form_l10_goals_against": float(goals_against),
                "form_l10_goal_difference": float(goal_difference),
                "form_l10_goals_per_match": float(goals_for / match_count),
                "form_l10_goals_against_per_match": float(goals_against / match_count),
                "form_l10_goal_difference_per_match": float(goal_difference / match_count),
                "form_l10_elo_change": float(elo_change) if pd.notna(elo_change) else np.nan,
                "form_l10_elo_change_per_match": float(recent["team_elo_delta"].mean()) if recent["team_elo_delta"].notna().any() else np.nan,
                "weighted_form_l10_result_score": weighted_mean(recent["actual_score"], weights),
                "weighted_form_l10_win_pct": weighted_mean(recent["win_indicator"], weights),
                "weighted_form_l10_goals_for_per_match": weighted_mean(recent["team_score"], weights),
                "weighted_form_l10_goals_against_per_match": weighted_mean(recent["opponent_score"], weights),
                "weighted_form_l10_goal_difference_per_match": weighted_mean(recent["goal_difference"], weights),
                "weighted_form_l10_elo_change_per_match": weighted_mean(recent["team_elo_delta"], weights),
                "form_l10_first_match_date": recent["date"].min(),
                "form_l10_last_match_date": recent["date"].max(),
            }
        )
    form_rows.append(form_row)

form_df = pd.DataFrame(form_rows)
outcome_df = outcome_df.merge(form_df, on=["edition", "country"], how="left")
outcome_df["edition_start_date"] = outcome_df["edition"].map(edition_start_dates)

form_feature_columns = [
    "form_l10_matches",
    "form_l10_win_pct",
    "form_l10_goals_for",
    "form_l10_goals_against",
    "form_l10_goal_difference",
    "form_l10_goals_per_match",
    "form_l10_goals_against_per_match",
    "form_l10_goal_difference_per_match",
    "form_l10_elo_change",
    "form_l10_elo_change_per_match",
    "weighted_form_l10_result_score",
    "weighted_form_l10_win_pct",
    "weighted_form_l10_goals_for_per_match",
    "weighted_form_l10_goals_against_per_match",
    "weighted_form_l10_goal_difference_per_match",
    "weighted_form_l10_elo_change_per_match",
]

pre_tournament_features = [
    "start_elo",
    "is_host",
    "prior_world_cup_participations",
    "prior_avg_finish_score",
    "prior_best_finish_score",
    "prior_avg_goals_per_match",
    "prior_avg_goal_diff_per_match",
    "previous_finish_score",
    "previous_position",
] + form_feature_columns

in_tournament_features = [
    "matches_played",
    "gs",
    "ga",
    "goal_difference",
    "goals_per_match",
    "goals_against_per_match",
    "goal_difference_per_match",
    "finish_elo",
    "elo_change",
]

def build_correlation_table(df, feature_columns, target="finish_score"):
    rows = []
    for feature in feature_columns:
        analysis_df = df[[feature, target]].dropna()
        rows.append(
            {
                "feature": feature,
                "n": len(analysis_df),
                "pearson_corr": analysis_df[feature].corr(analysis_df[target], method="pearson"),
                "spearman_corr": analysis_df[feature].corr(analysis_df[target], method="spearman"),
            }
        )
    corr_df = pd.DataFrame(rows)
    corr_df["abs_spearman_corr"] = corr_df["spearman_corr"].abs()
    return corr_df.sort_values("abs_spearman_corr", ascending=False).reset_index(drop=True)

pre_tournament_corr = build_correlation_table(outcome_df, pre_tournament_features)
in_tournament_corr = build_correlation_table(outcome_df, in_tournament_features)

# Sanity checks requested in the plan.
assert not outcome_df.duplicated(["edition", "country"]).any(), "Expected one row per edition + country."
assert np.allclose(outcome_df.loc[outcome_df["placement"].eq("Winner"), "finish_score"], 1.0), "Winners must score 1.0."
lowest_finish_by_edition = outcome_df.loc[outcome_df.groupby("edition")["position"].idxmax()]
assert np.allclose(lowest_finish_by_edition["finish_score"], 0.0), "Lowest position in each edition must score 0.0."

known_hosts = {
    (1966, "England"),
    (1998, "France"),
    (2010, "South Africa"),
    (2022, "Qatar"),
    (2002, "South Korea"),
    (2002, "Japan"),
}
missing_hosts = [
    f"{country} {edition}"
    for edition, country in known_hosts
    if outcome_df.loc[
        outcome_df["edition"].eq(edition) & outcome_df["country"].eq(country),
        "is_host",
    ].sum() != 1
]
assert not missing_hosts, f"Host flags missing for: {missing_hosts}"
assert outcome_df["form_l10_matches"].dropna().le(FORM_MATCH_WINDOW).all(), "Form windows must not exceed last-k size."
assert (
    outcome_df.loc[outcome_df["form_l10_last_match_date"].notna(), "form_l10_last_match_date"]
    < outcome_df.loc[outcome_df["form_l10_last_match_date"].notna(), "edition_start_date"]
).all(), "Form matches must be before the tournament start date."
assert not outcome_df.duplicated(["edition", "country"]).any(), "Expected no duplicate rows after adding form features."
excluded_correlation_features = {"edition", "country", "placement"}
assert excluded_correlation_features.isdisjoint(pre_tournament_features + in_tournament_features), "Correlation features include identifiers or categorical placement."
weighted_diff_check = outcome_df[
    outcome_df["form_l10_matches"].ge(2)
    & outcome_df["form_l10_win_pct"].notna()
    & outcome_df["weighted_form_l10_win_pct"].notna()
    & ~np.isclose(outcome_df["form_l10_win_pct"], outcome_df["weighted_form_l10_win_pct"])
]
assert not weighted_diff_check.empty, "Expected at least one mixed-form team where weighted and unweighted form differ."

spot_checks = outcome_df.set_index(["edition", "country"])
for edition, country in [(1966, "England"), (2022, "Argentina")]:
    spot_row = spot_checks.loc[(edition, country)]
    assert pd.notna(spot_row["form_l10_last_match_date"]), f"Missing form rows for {country} {edition}."
    assert spot_row["form_l10_last_match_date"] < spot_row["edition_start_date"], f"Form leakage for {country} {edition}."

print(f"Analysis rows: {len(outcome_df):,}")
print(f"Editions: {outcome_df['edition'].min()}-{outcome_df['edition'].max()} ({outcome_df['edition'].nunique()} tournaments)")

print("\nPre-tournament predictors vs finish_score")
display(pre_tournament_corr)

print("\nIn-tournament explanatory stats vs finish_score")
display(in_tournament_corr)

fig_pre = px.bar(
    pre_tournament_corr,
    x="spearman_corr",
    y="feature",
    orientation="h",
    color="spearman_corr",
    color_continuous_scale="RdBu",
    title="Pre-Tournament Feature Correlation with World Cup Finish Score",
    labels={"spearman_corr": "Spearman correlation", "feature": "Feature"},
)
fig_pre.update_layout(yaxis=dict(categoryorder="total ascending"), height=500, width=950)
fig_pre.show()

fig_in = px.bar(
    in_tournament_corr,
    x="spearman_corr",
    y="feature",
    orientation="h",
    color="spearman_corr",
    color_continuous_scale="RdBu",
    title="In-Tournament Stat Correlation with World Cup Finish Score",
    labels={"spearman_corr": "Spearman correlation", "feature": "Feature"},
)
fig_in.update_layout(yaxis=dict(categoryorder="total ascending"), height=500, width=950)
fig_in.show()

heatmap_columns = ["finish_score"] + pre_tournament_features 
heatmap_corr = outcome_df[heatmap_columns].corr(method="spearman")
fig_heatmap = px.imshow(
    heatmap_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Spearman Correlation Heatmap: Outcome, Predictors",
)
fig_heatmap.update_layout(height=1050, width=1150)
fig_heatmap.show()

heatmap_columns = ["finish_score"] + in_tournament_features
heatmap_corr = outcome_df[heatmap_columns].corr(method="spearman")
fig_heatmap = px.imshow(
    heatmap_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Spearman Correlation Heatmap: Outcome, Tournament Stats",
)
fig_heatmap.update_layout(height=900, width=1000)
fig_heatmap.show()

strongest_pre_features = pre_tournament_corr.head(3)["feature"].tolist()
for feature in strongest_pre_features:
    scatter_df = outcome_df[["edition", "country", "placement", "finish_score", feature]].dropna()
    fig_scatter = px.scatter(
        scatter_df,
        x=feature,
        y="finish_score",
        color="placement",
        hover_data=["country", "edition"],
        trendline="ols",
        trendline_scope="overall",
        title=f"{feature} vs World Cup Finish Score",
        labels={feature: feature.replace("_", " ").title(), "finish_score": "Finish Score"},
    )
    fig_scatter.update_layout(height=550, width=900)
    fig_scatter.show()

# Show the analysis dataframe so it can be reused in later cells.
display(outcome_df.head(20))


Analysis rows: 489
Editions: 1930-2022 (22 tournaments)

Pre-tournament predictors vs finish_score


,feature,n,pearson_corr,spearman_corr,abs_spearman_corr
0,start_elo,477,0.471946,0.477454,0.477454
1,prior_best_finish_score,405,0.359745,0.384366,0.384366
2,prior_avg_goal_diff_per_match,405,0.320866,0.351370,0.351370
3,prior_avg_goals_per_match,405,0.294832,0.349616,0.349616
4,weighted_form_l10_goal_difference_per_match,487,0.349838,0.347870,0.347870
5,prior_world_cup_participations,489,0.374668,0.338899,0.338899
6,form_l10_goal_difference,487,0.337740,0.330836,0.330836
7,form_l10_goal_difference_per_match,487,0.331910,0.328792,0.328792
8,prior_avg_finish_score,405,0.298615,0.326071,0.326071
9,weighted_form_l10_result_score,487,0.313861,0.313007,0.313007



In-tournament explanatory stats vs finish_score


,feature,n,pearson_corr,spearman_corr,abs_spearman_corr
0,matches_played,489,0.861094,0.897085,0.897085
1,gs,489,0.782187,0.809571,0.809571
2,goal_difference_per_match,489,0.631593,0.722577,0.722577
3,goal_difference,489,0.705068,0.708284,0.708284
4,finish_elo,477,0.662950,0.666952,0.666952
5,elo_change,477,0.601942,0.586274,0.586274
6,goals_per_match,489,0.524178,0.574395,0.574395
7,goals_against_per_match,489,-0.412742,-0.447514,0.447514
8,ga,489,0.074316,0.077908,0.077908


,tournament_id,year,country,team_id,team_code,confederation,placement,position,matches_played,gs,ga,start_elo,finish_elo,elo_change,next_edition,next_placement,next_position,edition,edition_team_count,edition_finish_scale,finish_score,placement_score,goal_difference,goals_per_match,goals_against_per_match,goal_difference_per_match,is_host,prior_world_cup_participations,previous_finish_score,previous_position,prior_avg_finish_score,prior_best_finish_score,prior_avg_goals_per_match,prior_avg_goal_diff_per_match,form_l10_matches,form_l10_win_pct,form_l10_goals_for,form_l10_goals_against,form_l10_goal_difference,form_l10_goals_per_match,form_l10_goals_against_per_match,form_l10_goal_difference_per_match,form_l10_elo_change,form_l10_elo_change_per_match,weighted_form_l10_result_score,weighted_form_l10_win_pct,weighted_form_l10_goals_for_per_match,weighted_form_l10_goals_against_per_match,weighted_form_l10_goal_difference_per_match,weighted_form_l10_elo_change_per_match,form_l10_first_match_date,form_l10_last_match_date,edition_start_date
0,WC-1982,1982,Algeria,DZA,DZA,CAF,Group Stage,24,3,5,5,1696.0,1713.0,17.0,1986.0,Group Stage,24.0,1982,24,24,0.000000,1,0,1.666667,1.666667,0.000000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,10.0,0.6,13.0,8.0,5.0,1.3,0.8,0.5,2.0,0.222222,0.590909,0.454545,1.200000,0.963636,0.236364,-9.222222,1981-10-10,1982-04-28,1982-06-13
1,WC-1986,1986,Algeria,DZA,DZA,CAF,Group Stage,24,3,1,5,1618.0,1609.0,-9.0,2010.0,Group Stage,32.0,1986,24,24,0.000000,1,-4,0.333333,1.666667,-1.333333,0,1,0.000000,24.0,0.000000,0.000000,1.666667,0.000000,10.0,0.1,9.0,14.0,-5.0,0.9,1.4,-0.5,-69.0,-7.666667,0.272727,0.090909,0.927273,1.418182,-0.490909,-7.644444,1985-12-11,1986-05-11,1986-05-31
2,WC-2010,2010,Algeria,DZA,DZA,CAF,Group Stage,32,3,0,2,1539.0,1535.0,-4.0,2014.0,Round of 16,16.0,2010,32,32,0.000000,1,-2,0.000000,0.666667,-0.666667,0,2,0.000000,24.0,0.000000,0.000000,1.000000,-0.666667,10.0,0.4,6.0,16.0,-10.0,0.6,1.6,-1.0,-7.0,-0.777778,0.381818,0.345455,0.527273,1.781818,-1.254545,-1.260870,2009-11-18,2010-06-05,2010-06-11
3,WC-2014,2014,Algeria,DZA,DZA,CAF,Round of 16,16,4,7,7,1625.0,1673.0,48.0,NaN,NaN,NaN,2014,32,32,0.516129,2,0,1.750000,1.750000,0.000000,0,3,0.000000,32.0,0.000000,0.000000,0.666667,-0.666667,10.0,0.8,19.0,8.0,11.0,1.9,0.8,1.1,102.0,10.200000,0.854545,0.818182,1.927273,0.854545,1.072727,9.818182,2013-06-02,2014-06-04,2014-06-12
4,WC-2006,2006,Angola,AGO,AGO,CAF,Group Stage,32,3,1,2,1565.0,1595.0,30.0,NaN,NaN,NaN,2006,32,32,0.000000,1,-1,0.333333,0.666667,-0.333333,0,0,NaN,NaN,NaN,NaN,NaN,NaN,10.0,0.3,16.0,16.0,0.0,1.6,1.6,0.0,20.0,2.000000,0.418182,0.363636,1.836364,1.690909,0.145455,2.672727,2005-11-16,2006-06-02,2006-06-09
5,WC-1930,1930,Argentina,ARG,ARG,CONMEBOL,Runner-up,2,5,18,9,2073.0,2059.0,-14.0,1934.0,Round of 16,16.0,1930,13,13,0.916667,6,9,3.600000,1.800000,1.800000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,10.0,0.5,17.0,7.0,10.0,1.7,0.7,1.0,43.0,4.300000,0.709091,0.509091,1.836364,0.654545,1.181818,3.272727,1928-08-30,1930-05-25,1930-07-13
6,WC-1934,1934,Argentina,ARG,ARG,CONMEBOL,Round of 16,16,1,2,3,2062.0,2010.0,-52.0,1958.0,Group Stage,16.0,1934,16,16,0.000000,2,-1,2.000000,3.000000,-1.000000,0,1,0.916667,2.0,0.916667,0.916667,3.600000,1.800000,10.0,0.6,17.0,8.0,9.0,1.7,0.8,0.9,3.0,0.300000,0.663636,0.600000,1.690909,0.818182,0.872727,1.581818,1930-08-03,1933-12-14,1934-05-27
7,WC-1958,1958,Argentina,ARG,ARG,CONMEBOL,Group Stage,16,3,5,10,1984.0,1922.0,-62.0,1962.0,Group Stage,16.0,1958,16,16,0.000000,1,-5,1.666667,3.333333,-1.666667,0,2,0.000000,16.0,0.458333,0.916667,2.800000,0.400000,10.0,0.6,16.0,7.0,9.0,1.6,0.7,0.9,-64.0,-6.400000,0.636364,0.636364,1.672727,0.472727,1.200000,-3.945455,1957-07-07,1958-04-30,1958-06-08
8,WC-1962,1962,Argentina,ARG,ARG,CONMEBOL,Group Stage,16,3,2,3,1980.0,1941.0,-39.0,1966.0,Quarter-final,8.0,1962,16,16,0.000000,1,-1,0.666667,1.000000,-0.333333,0,3,0.000000,16.0,0.305556,0.916667,2.422222,-0.288889,10.0,0.3,14.0,13.0,1.0,1.4,1.3,0.1,-36.0,-3.600000,0.536364,0.345455,1.54

In [246]:
# Last-k World Cup history correlation analysis
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

WORLD_CUP_LOOKBACK = 5

# Reuse the previous analysis dataframe when available; rebuild it when this cell is run by itself.
if "outcome_df" not in globals():
    placement_score_map = {
        "Winner": 7,
        "Runner-up": 6,
        "Third Place": 5,
        "Fourth Place": 4,
        "Semi-final": 4,
        "Quarter-final": 3,
        "Round of 16": 2,
        "Group Stage": 1,
    }
    placement_files = sorted(Path("data/processed/world_cup").glob("[0-9][0-9][0-9][0-9]/placement.csv"))
    placement_frames = []
    for placement_file in placement_files:
        edition = int(placement_file.parent.name)
        edition_df = pd.read_csv(placement_file)
        edition_df["edition"] = edition
        placement_frames.append(edition_df)
    outcome_df = pd.concat(placement_frames, ignore_index=True)
    outcome_df = outcome_df.sort_values(["country", "edition"]).reset_index(drop=True)
    outcome_df["edition_team_count"] = outcome_df.groupby("edition")["country"].transform("count")
    outcome_df["position"] = pd.to_numeric(outcome_df["position"], errors="coerce")
    outcome_df["edition_finish_scale"] = outcome_df.groupby("edition")["position"].transform("max")
    outcome_df["edition_finish_scale"] = outcome_df[["edition_team_count", "edition_finish_scale"]].max(axis=1)
    outcome_df["finish_score"] = 1 - (
        (outcome_df["position"] - 1) / (outcome_df["edition_finish_scale"] - 1)
    )
    outcome_df["placement_score"] = outcome_df["placement"].map(placement_score_map)
    for column in ["matches_played", "gs", "ga", "start_elo", "finish_elo", "elo_change"]:
        outcome_df[column] = pd.to_numeric(outcome_df[column], errors="coerce")
    outcome_df["goal_difference"] = outcome_df["gs"] - outcome_df["ga"]
    outcome_df["goals_per_match"] = outcome_df["gs"] / outcome_df["matches_played"]
    outcome_df["goals_against_per_match"] = outcome_df["ga"] / outcome_df["matches_played"]
    outcome_df["goal_difference_per_match"] = outcome_df["goal_difference"] / outcome_df["matches_played"]

wc_history_base = outcome_df.copy()
for column in [
    "position",
    "finish_score",
    "matches_played",
    "gs",
    "ga",
    "goal_difference",
    "goals_per_match",
    "goals_against_per_match",
    "goal_difference_per_match",
    "elo_change",
]:
    wc_history_base[column] = pd.to_numeric(wc_history_base[column], errors="coerce")

all_editions = sorted(wc_history_base["edition"].dropna().astype(int).unique().tolist())
team_edition_lookup = {
    (int(row.edition), str(row.country)): row
    for row in wc_history_base.itertuples(index=False)
}

def safe_mean(values):
    values = pd.to_numeric(pd.Series(values), errors="coerce")
    valid = values.dropna()
    return float(valid.mean()) if not valid.empty else np.nan

def safe_sum(values):
    values = pd.to_numeric(pd.Series(values), errors="coerce")
    valid = values.dropna()
    return float(valid.sum()) if not valid.empty else np.nan

def weighted_mean(values, weights):
    values = pd.to_numeric(pd.Series(values), errors="coerce")
    weights = pd.Series(weights, index=values.index, dtype=float)
    valid = values.notna()
    if not valid.any():
        return np.nan
    return float(np.average(values[valid], weights=weights[valid]))

wc_l5_rows = []
prior_edition_audit_rows = []
for row in wc_history_base[["edition", "country"]].itertuples(index=False):
    edition = int(row.edition)
    country = str(row.country)
    prior_editions = [prior_edition for prior_edition in all_editions if prior_edition < edition][-WORLD_CUP_LOOKBACK:]
    weights = pd.Series(np.arange(1, len(prior_editions) + 1, dtype=float))

    appeared_flags = []
    finish_scores_with_dnq = []
    appearance_positions = []
    appearance_goals_for = []
    appearance_goals_against = []
    appearance_goal_difference = []
    appearance_goals_per_match = []
    appearance_goals_against_per_match = []
    appearance_goal_difference_per_match = []
    appearance_elo_change = []

    for prior_edition in prior_editions:
        prior_row = team_edition_lookup.get((prior_edition, country))
        appeared = prior_row is not None
        appeared_flags.append(1.0 if appeared else 0.0)
        finish_scores_with_dnq.append(float(getattr(prior_row, "finish_score")) if appeared else 0.0)
        prior_edition_audit_rows.append(
            {
                "edition": edition,
                "country": country,
                "prior_edition": int(prior_edition),
                "appeared": int(appeared),
            }
        )
        if appeared:
            appearance_positions.append(getattr(prior_row, "position"))
            appearance_goals_for.append(getattr(prior_row, "gs"))
            appearance_goals_against.append(getattr(prior_row, "ga"))
            appearance_goal_difference.append(getattr(prior_row, "goal_difference"))
            appearance_goals_per_match.append(getattr(prior_row, "goals_per_match"))
            appearance_goals_against_per_match.append(getattr(prior_row, "goals_against_per_match"))
            appearance_goal_difference_per_match.append(getattr(prior_row, "goal_difference_per_match"))
            appearance_elo_change.append(getattr(prior_row, "elo_change"))

    appearances = int(sum(appeared_flags))
    prior_count = len(prior_editions)
    total_weight = float(weights.sum()) if prior_count else np.nan
    weighted_appearance_rate = (
        float(np.dot(appeared_flags, weights) / total_weight) if prior_count else np.nan
    )
    weighted_finish_score = (
        float(np.dot(finish_scores_with_dnq, weights) / total_weight) if prior_count else np.nan
    )

    appeared_weights = [weight for flag, weight in zip(appeared_flags, weights, strict=False) if flag]

    wc_l5_rows.append(
        {
            "edition": edition,
            "country": country,
            "wc_l5_prior_edition_count": int(prior_count),
            "wc_l5_prior_editions": ",".join(str(prior_edition) for prior_edition in prior_editions),
            "wc_l5_appearances": appearances,
            "wc_l5_appearance_rate": float(appearances / prior_count) if prior_count else 0.0,
            "wc_l5_avg_finish_score": float(np.mean(finish_scores_with_dnq)) if prior_count else np.nan,
            "wc_l5_best_finish_score": float(np.max(finish_scores_with_dnq)) if prior_count else np.nan,
            "wc_l5_avg_position": safe_mean(appearance_positions),
            "wc_l5_goals_for": safe_sum(appearance_goals_for),
            "wc_l5_goals_against": safe_sum(appearance_goals_against),
            "wc_l5_goal_difference": safe_sum(appearance_goal_difference),
            "wc_l5_goals_per_match": safe_mean(appearance_goals_per_match),
            "wc_l5_goals_against_per_match": safe_mean(appearance_goals_against_per_match),
            "wc_l5_goal_difference_per_match": safe_mean(appearance_goal_difference_per_match),
            "wc_l5_elo_change": safe_sum(appearance_elo_change),
            "wc_l5_elo_change_per_appearance": safe_mean(appearance_elo_change),
            "weighted_wc_l5_appearance_rate": weighted_appearance_rate,
            "weighted_wc_l5_finish_score": weighted_finish_score,
            "weighted_wc_l5_position": weighted_mean(appearance_positions, appeared_weights) if appearance_positions else np.nan,
            "weighted_wc_l5_goals_per_match": weighted_mean(appearance_goals_per_match, appeared_weights) if appearance_goals_per_match else np.nan,
            "weighted_wc_l5_goals_against_per_match": weighted_mean(appearance_goals_against_per_match, appeared_weights) if appearance_goals_against_per_match else np.nan,
            "weighted_wc_l5_goal_difference_per_match": weighted_mean(appearance_goal_difference_per_match, appeared_weights) if appearance_goal_difference_per_match else np.nan,
            "weighted_wc_l5_elo_change_per_appearance": weighted_mean(appearance_elo_change, appeared_weights) if appearance_elo_change else np.nan,
        }
    )

wc_l5_features = [
    "wc_l5_appearances",
    "wc_l5_appearance_rate",
    "wc_l5_avg_finish_score",
    "wc_l5_best_finish_score",
    "wc_l5_avg_position",
    "wc_l5_goals_for",
    "wc_l5_goals_against",
    "wc_l5_goal_difference",
    "wc_l5_goals_per_match",
    "wc_l5_goals_against_per_match",
    "wc_l5_goal_difference_per_match",
    "wc_l5_elo_change",
    "wc_l5_elo_change_per_appearance",
    "weighted_wc_l5_appearance_rate",
    "weighted_wc_l5_finish_score",
    "weighted_wc_l5_position",
    "weighted_wc_l5_goals_per_match",
    "weighted_wc_l5_goals_against_per_match",
    "weighted_wc_l5_goal_difference_per_match",
    "weighted_wc_l5_elo_change_per_appearance",
]

wc_l5_feature_df = pd.DataFrame(wc_l5_rows)
wc_l5_analysis_df = wc_history_base.merge(wc_l5_feature_df, on=["edition", "country"], how="left")
wc_l5_prior_edition_audit_df = pd.DataFrame(prior_edition_audit_rows)

# Mirror the first correlation-analysis cell's baseline predictors and explanatory stats.
if "is_host" not in wc_l5_analysis_df.columns:
    host_pairs = set(
        pd.read_csv("data/processed/world_cup/all_editions/results.csv")[["edition", "country"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )
    wc_l5_analysis_df["is_host"] = [
        int((int(edition), country) in host_pairs)
        for edition, country in zip(wc_l5_analysis_df["edition"], wc_l5_analysis_df["country"])
    ]

team_history = wc_l5_analysis_df.sort_values(["country", "edition"]).groupby("country", group_keys=False)
if "prior_world_cup_participations" not in wc_l5_analysis_df.columns:
    wc_l5_analysis_df["prior_world_cup_participations"] = team_history.cumcount()
if "previous_finish_score" not in wc_l5_analysis_df.columns:
    wc_l5_analysis_df["previous_finish_score"] = team_history["finish_score"].shift(1)
if "previous_position" not in wc_l5_analysis_df.columns:
    wc_l5_analysis_df["previous_position"] = team_history["position"].shift(1)
if "prior_avg_finish_score" not in wc_l5_analysis_df.columns:
    wc_l5_analysis_df["prior_avg_finish_score"] = team_history["finish_score"].transform(
        lambda series: series.shift(1).expanding().mean()
    )
if "prior_best_finish_score" not in wc_l5_analysis_df.columns:
    wc_l5_analysis_df["prior_best_finish_score"] = team_history["finish_score"].transform(
        lambda series: series.shift(1).expanding().max()
    )
if "prior_avg_goals_per_match" not in wc_l5_analysis_df.columns:
    wc_l5_analysis_df["prior_avg_goals_per_match"] = team_history["goals_per_match"].transform(
        lambda series: series.shift(1).expanding().mean()
    )
if "prior_avg_goal_diff_per_match" not in wc_l5_analysis_df.columns:
    wc_l5_analysis_df["prior_avg_goal_diff_per_match"] = team_history["goal_difference_per_match"].transform(
        lambda series: series.shift(1).expanding().mean()
    )

baseline_pre_tournament_features = [
    "start_elo",
    "is_host",
    "prior_world_cup_participations",
    "prior_avg_finish_score",
    "prior_best_finish_score",
    "prior_avg_goals_per_match",
    "prior_avg_goal_diff_per_match",
    "previous_finish_score",
    "previous_position",
]

wc_l5_pre_tournament_features = baseline_pre_tournament_features + wc_l5_features

in_tournament_features = [
    "matches_played",
    "gs",
    "ga",
    "goal_difference",
    "goals_per_match",
    "goals_against_per_match",
    "goal_difference_per_match",
    "finish_elo",
    "elo_change",
]

def build_correlation_table(df, feature_columns, target="finish_score"):
    rows = []
    for feature in feature_columns:
        analysis_df = df[[feature, target]].dropna()
        rows.append(
            {
                "feature": feature,
                "n": len(analysis_df),
                "pearson_corr": analysis_df[feature].corr(analysis_df[target], method="pearson"),
                "spearman_corr": analysis_df[feature].corr(analysis_df[target], method="spearman"),
            }
        )
    corr_df = pd.DataFrame(rows)
    corr_df["abs_spearman_corr"] = corr_df["spearman_corr"].abs()
    return corr_df.sort_values("abs_spearman_corr", ascending=False).reset_index(drop=True)

wc_l5_corr = build_correlation_table(wc_l5_analysis_df, wc_l5_pre_tournament_features)
in_tournament_corr = build_correlation_table(wc_l5_analysis_df, in_tournament_features)

# Sanity checks for leakage and feature integrity.
assert not wc_l5_analysis_df.duplicated(["edition", "country"]).any(), "Expected one row per edition + country."
assert wc_l5_analysis_df["wc_l5_prior_edition_count"].le(WORLD_CUP_LOOKBACK).all(), "Prior edition window is too large."
if not wc_l5_prior_edition_audit_df.empty:
    assert (wc_l5_prior_edition_audit_df["prior_edition"] < wc_l5_prior_edition_audit_df["edition"]).all(), "Prior editions must be before the current edition."
assert wc_l5_analysis_df.loc[wc_l5_analysis_df["edition"].eq(1930), "wc_l5_appearances"].eq(0).all(), "First-edition rows must have no prior appearances."
for edition, country in [(2022, "Brazil"), (2018, "Germany")]:
    audit = wc_l5_prior_edition_audit_df[
        wc_l5_prior_edition_audit_df["edition"].eq(edition)
        & wc_l5_prior_edition_audit_df["country"].eq(country)
    ]
    assert not audit.empty, f"Missing prior-edition audit rows for {country} {edition}."
    assert audit["prior_edition"].lt(edition).all(), f"Current edition leaked into {country} {edition}."
weighted_diff_check = wc_l5_analysis_df[
    wc_l5_analysis_df["wc_l5_prior_edition_count"].ge(2)
    & wc_l5_analysis_df["wc_l5_avg_finish_score"].notna()
    & wc_l5_analysis_df["weighted_wc_l5_finish_score"].notna()
    & ~np.isclose(wc_l5_analysis_df["wc_l5_avg_finish_score"], wc_l5_analysis_df["weighted_wc_l5_finish_score"])
]
assert not weighted_diff_check.empty, "Expected weighted and unweighted last-5 finish scores to differ somewhere."
excluded_correlation_features = {"edition", "country", "placement"}
assert excluded_correlation_features.isdisjoint(wc_l5_pre_tournament_features + in_tournament_features), "Correlation features include identifiers or categorical placement."

print(f"Last-{WORLD_CUP_LOOKBACK} World Cup analysis rows: {len(wc_l5_analysis_df):,}")
print("\nLast-5 World Cup history features vs finish_score")
display(wc_l5_corr)

fig_wc_l5 = px.bar(
    wc_l5_corr,
    x="spearman_corr",
    y="feature",
    orientation="h",
    color="spearman_corr",
    color_continuous_scale="RdBu",
    title=f"Pre-Tournament Predictors + Last-{WORLD_CUP_LOOKBACK} World Cup History Correlation with Finish Score",
    labels={"spearman_corr": "Spearman correlation", "feature": "Feature"},
)
fig_wc_l5.update_layout(yaxis=dict(categoryorder="total ascending"), height=650, width=1050)
fig_wc_l5.show()

heatmap_columns = ["finish_score"] + wc_l5_pre_tournament_features
heatmap_corr = wc_l5_analysis_df[heatmap_columns].corr(method="spearman")
fig_heatmap = px.imshow(
    heatmap_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title=f"Spearman Correlation Heatmap: Outcome, Baseline Predictors, and Last-{WORLD_CUP_LOOKBACK} World Cup History",
)
fig_heatmap.update_layout(height=1100, width=1150)
fig_heatmap.show()

fig_in = px.bar(
    in_tournament_corr,
    x="spearman_corr",
    y="feature",
    orientation="h",
    color="spearman_corr",
    color_continuous_scale="RdBu",
    title="In-Tournament Stat Correlation with World Cup Finish Score",
    labels={"spearman_corr": "Spearman correlation", "feature": "Feature"},
)
fig_in.update_layout(yaxis=dict(categoryorder="total ascending"), height=500, width=950)
fig_in.show()

in_heatmap_columns = ["finish_score"] + in_tournament_features
in_heatmap_corr = wc_l5_analysis_df[in_heatmap_columns].corr(method="spearman")
fig_in_heatmap = px.imshow(
    in_heatmap_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Spearman Correlation Heatmap: Outcome and Tournament Stats",
)
fig_in_heatmap.update_layout(height=900, width=1000)
fig_in_heatmap.show()

strongest_wc_l5_features = wc_l5_corr.head(3)["feature"].tolist()
for feature in strongest_wc_l5_features:
    scatter_df = wc_l5_analysis_df[["edition", "country", "placement", "finish_score", feature]].dropna()
    fig_scatter = px.scatter(
        scatter_df,
        x=feature,
        y="finish_score",
        color="placement",
        hover_data=["country", "edition"],
        trendline="ols",
        trendline_scope="overall",
        title=f"{feature} vs World Cup Finish Score",
        labels={feature: feature.replace("_", " ").title(), "finish_score": "Finish Score"},
    )
    fig_scatter.update_layout(height=550, width=900)
    fig_scatter.show()

display(wc_l5_analysis_df.head(20))


Last-5 World Cup analysis rows: 489

Last-5 World Cup history features vs finish_score


,feature,n,pearson_corr,spearman_corr,abs_spearman_corr
0,start_elo,477,0.471946,0.477454,0.477454
1,wc_l5_goal_difference,382,0.391735,0.424054,0.424054
2,wc_l5_goals_for,382,0.394838,0.396696,0.396696
3,weighted_wc_l5_finish_score,476,0.393595,0.387492,0.387492
4,wc_l5_avg_finish_score,476,0.389561,0.384460,0.384460
5,prior_best_finish_score,405,0.359745,0.384366,0.384366
6,weighted_wc_l5_goal_difference_per_match,382,0.365913,0.383999,0.383999
7,wc_l5_goal_difference_per_match,382,0.366303,0.382578,0.382578
8,prior_avg_goal_diff_per_match,405,0.320866,0.351370,0.351370
9,prior_avg_goals_per_match,405,0.294832,0.349616,0.349616


,tournament_id,year,country,team_id,team_code,confederation,placement,position,matches_played,gs,ga,start_elo,finish_elo,elo_change,next_edition,next_placement,next_position,edition,edition_team_count,edition_finish_scale,finish_score,placement_score,goal_difference,goals_per_match,goals_against_per_match,goal_difference_per_match,is_host,prior_world_cup_participations,previous_finish_score,previous_position,prior_avg_finish_score,prior_best_finish_score,prior_avg_goals_per_match,prior_avg_goal_diff_per_match,form_l10_matches,form_l10_win_pct,form_l10_goals_for,form_l10_goals_against,form_l10_goal_difference,form_l10_goals_per_match,form_l10_goals_against_per_match,form_l10_goal_difference_per_match,form_l10_elo_change,form_l10_elo_change_per_match,weighted_form_l10_result_score,weighted_form_l10_win_pct,weighted_form_l10_goals_for_per_match,weighted_form_l10_goals_against_per_match,weighted_form_l10_goal_difference_per_match,weighted_form_l10_elo_change_per_match,form_l10_first_match_date,form_l10_last_match_date,edition_start_date,wc_l5_prior_edition_count,wc_l5_prior_editions,wc_l5_appearances,wc_l5_appearance_rate,wc_l5_avg_finish_score,wc_l5_best_finish_score,wc_l5_avg_position,wc_l5_goals_for,wc_l5_goals_against,wc_l5_goal_difference,wc_l5_goals_per_match,wc_l5_goals_against_per_match,wc_l5_goal_difference_per_match,wc_l5_elo_change,wc_l5_elo_change_per_appearance,weighted_wc_l5_appearance_rate,weighted_wc_l5_finish_score,weighted_wc_l5_position,weighted_wc_l5_goals_per_match,weighted_wc_l5_goals_against_per_match,weighted_wc_l5_goal_difference_per_match,weighted_wc_l5_elo_change_per_appearance
0,WC-1982,1982,Algeria,DZA,DZA,CAF,Group Stage,24,3,5,5,1696.0,1713.0,17.0,1986.0,Group Stage,24.0,1982,24,24,0.000000,1,0,1.666667,1.666667,0.000000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,10.0,0.6,13.0,8.0,5.0,1.3,0.8,0.5,2.0,0.222222,0.590909,0.454545,1.200000,0.963636,0.236364,-9.222222,1981-10-10,1982-04-28,1982-06-13,5,"1962,1966,1970,1974,1978",0,0.0,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
1,WC-1986,1986,Algeria,DZA,DZA,CAF,Group Stage,24,3,1,5,1618.0,1609.0,-9.0,2010.0,Group Stage,32.0,1986,24,24,0.000000,1,-4,0.333333,1.666667,-1.333333,0,1,0.000000,24.0,0.000000,0.000000,1.666667,0.000000,10.0,0.1,9.0,14.0,-5.0,0.9,1.4,-0.5,-69.0,-7.666667,0.272727,0.090909,0.927273,1.418182,-0.490909,-7.644444,1985-12-11,1986-05-11,1986-05-31,5,"1966,1970,1974,1978,1982",1,0.2,0.000000,0.000000,24.000000,5.0,5.0,0.0,1.666667,1.666667,0.000000,17.0,17.000000,0.333333,0.000000,24.000000,1.666667,1.666667,0.000000,17.000000
2,WC-2010,2010,Algeria,DZA,DZA,CAF,Group Stage,32,3,0,2,1539.0,1535.0,-4.0,2014.0,Round of 16,16.0,2010,32,32,0.000000,1,-2,0.000000,0.666667,-0.666667,0,2,0.000000,24.0,0.000000,0.000000,1.000000,-0.666667,10.0,0.4,6.0,16.0,-10.0,0.6,1.6,-1.0,-7.0,-0.777778,0.381818,0.345455,0.527273,1.781818,-1.254545,-1.260870,2009-11-18,2010-06-05,2010-06-11,5,"1990,1994,1998,2002,2006",0,0.0,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
3,WC-2014,2014,Algeria,DZA,DZA,CAF,Round of 16,16,4,7,7,1625.0,1673.0,48.0,NaN,NaN,NaN,2014,32,32,0.516129,2,0,1.750000,1.750000,0.000000,0,3,0.000000,32.0,0.000000,0.000000,0.666667,-0.666667,10.0,0.8,19.0,8.0,11.0,1.9,0.8,1.1,102.0,10.200000,0.854545,0.818182,1.927273,0.854545,1.072727,9.818182,2013-06-02,2014-06-04,2014-06-12,5,"1994,1998,2002,2006,2010",1,0.2,0.000000,0.000000,32.000000,0.0,2.0,-2.0,0.000000,0.666667,-0.666667,-4.0,-4.000000,0.333333,0.000000,32.000000,0.000000,0.666667,-0.666667,-4.000000
4,WC-2006,2006,Angola,AGO,AGO,CAF,Group Stage,32,3,1,2,1565.0,1595.0,30.0,NaN,NaN,NaN,2006,32,32,0.000000,1,-1,0.333333,0.666667,-0.333333,0,0,NaN,NaN,NaN,NaN,NaN,NaN,10.0,0.3,16.0,16.0,0.0,1.6,1.6,0.0,20.0,2.000000,0.418182,0.363636,1.836364,1.690909,0.145455,2.672727,2005-11-16,2006-06-02,2006-06-09,5,"1986,1990,1994,1998,2002",0,0.0,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,Na